In [3]:
import re
import numpy as np
import pandas as pd
import os
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../notebooks")
print(os.listdir(DATA_DIR))
members = pd.read_csv(
    DATA_DIR / "members.csv",
    parse_dates=["join_date"]
)

renewals = pd.read_csv(
    DATA_DIR / "renewal_opportunities.csv",
    parse_dates=["expiry_date", "renewal_deadline"]
)

events = pd.read_csv(
    DATA_DIR / "member_events.csv",
    parse_dates=["event_time"]
)

# Date columns supplied for the exercises.
members["join_year"] = members["join_date"].dt.year
renewals["expiry_year"] = renewals["expiry_date"].dt.year
renewals["expiry_month"] = renewals["expiry_date"].dt.month

['DATA_DICTIONARY.md', 'Kalpa_Plus_Student.ipynb', 'members.csv', 'renewal_opportunities.csv', 'member_events.csv']


1
Using members, group by acquisition_channel.

Write a function that receives the annual_fee Series and returns the difference between its highest and lowest values. Produce one row per channel with an output column named fee_range.

In [4]:
members.head()

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023


In [ ]:
def get_fee_range(annual_fees):
    return annual_fees.max() - annual_fees.min()

members_q1 = members.copy()
fee_range = (
    members_q1.groupby('acquisition_channel')['annual_fee']
    .apply(get_fee_range)
    .reset_index(name="fee_range")
)
fee_range

2.
Using members, group by city.

The function should receive the group DataFrame. Among members who were "PAID" on 9 October 2025, calculate the percentage who are "EXPIRED" on 9 October 2026.

Return one scalar per city, named expired_previous_paid_pct. If a city has no previously paid members, return NaN. Members previously marked "NOT_JOINED" or "EXPIRED" must not enter the denominator.

In [ ]:
def get_expired_percentage(city_members):
    previously_paid = city_members['status_2025_10_09'] == "PAID"
    denominator = previously_paid.sum()

    if denominator == 0:
        return np.nan

    expired_now = city_members['status_2026_10_09'] == "EXPIRED"
    return (expired_now & previously_paid).sum() / denominator * 100

In [ ]:
members_q2 = members.copy()
expired_previous_paid_pct = (
    members_q2.groupby('city')[['status_2025_10_09', 'status_2026_10_09']]
    .apply(get_expired_percentage)
    .reset_index(name="expired_previous_paid_pct")
)
expired_previous_paid_pct

3
Using members, group by acquisition_channel.

Write a function that receives the group DataFrame and returns the mean annual_fee of members currently marked "PAID". Produce current_paid_average_fee, with one row per acquisition channel.

In [ ]:
def get_current_paid_average_fee(channel_members):
    currently_paid = channel_members.loc[
        channel_members['status_2026_10_09'] == "PAID", 'annual_fee'
    ]

    if currently_paid.empty:
        return np.nan

    return currently_paid.mean()

members_q3 = members.copy()
current_paid_average_fee = (
    members_q3.groupby('acquisition_channel')[['status_2026_10_09', 'annual_fee']]
    .apply(get_current_paid_average_fee)
    .reset_index(name="current_paid_average_fee")
)
current_paid_average_fee

4
Using members, group by city.

Write a function that returns a named summary Series containing:
- registered_members: number of rows in the group.
- currently_paid: number currently marked "PAID".
- current_paid_pct: currently paid members as a percentage of all registered members in that city.

The final output should have one row per city and these three columns. This percentage is a billing-snapshot measure, not a renewal rate.

In [9]:
def summarize_city_members(city_members):
    currently_paid = city_members['status_2026_10_09'] == "PAID"
    registered_members = len(city_members)
    paid_count = currently_paid.sum()

    
    city_members["registered_members"]= registered_members
    city_members["currently_paid"]= paid_count
    city_members["current_paid_pct"]= round(paid_count / registered_members * 100,2)
    

city_paid_summary = (
    members.groupby('city')[['status_2026_10_09']]
    .apply(summarize_city_members)
    .reset_index()
)
city_paid_summary

,city,registered_members,currently_paid,current_paid_pct
0,Ahmedabad,1021.0,725.0,71.01
1,Bengaluru,1064.0,771.0,72.46
2,Delhi,1071.0,754.0,70.40
3,Gurugram,1055.0,755.0,71.56
4,Hyderabad,1124.0,789.0,70.20
5,Jaipur,1129.0,778.0,68.91
6,Mumbai,1076.0,754.0,70.07
7,Pune,1040.0,698.0,67.12


5
Using members, group by city and pass the annual_fee Series to your function.

Return a Series showing each member's annual fee as a percentage of the sum of listed annual fees in that city. Assign it back to members as city_fee_share_pct.

Preserve every original row index. This is a share of listed fees, not a share of collected revenue. If a total is zero, return zero for every row in that group.

In [11]:
def get_city_fee_share(annual_fees):
    total_fees = annual_fees.sum()
    if total_fees == 0:
        return annual_fees * 0

    return annual_fees / total_fees * 100

members_q5 = members.copy()
city_fee_share = (
    members.groupby('city', group_keys=False)['annual_fee']
    .apply(get_city_fee_share)
)
members.loc[city_fee_share.index, 'city_fee_share_pct'] = city_fee_share
members_q5

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year,city_fee_share_pct
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023,0.079970
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025,0.128952
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023,0.077350
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025,0.078587
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023,0.072860
...,...,...,...,...,...,...,...,...,...,...
8575,MP008500,2026-02-08,PLUS_STANDARD,1499,APP,Hyderabad,NOT_JOINED,PAID,2026,0.073305
8576,MP002065,2023-09-06,PLUS_STANDARD,1499,APP,Bengaluru,PAID,PAID,2023,0.077350
8577,MP007986,2026-03-11,PLUS_STANDARD,1499,WEB,Pune,NOT_JOINED,PAID,2026,0.080291
8578,MP002304,2023-09-08,PLUS_PREMIUM,2499,WEB,Gurugram,PAID,PAID,2023,0.133319
